In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import functions as F

In [0]:
dbutils.widgets.text("catalogo", "catalog_dev")
dbutils.widgets.text("esquema_source", "bronze")
dbutils.widgets.text("esquema_sink", "silver")

In [0]:
catalogo = dbutils.widgets.get("catalogo")
esquema_source = dbutils.widgets.get("esquema_source")
esquema_sink = dbutils.widgets.get("esquema_sink")

### UDF

In [0]:
def estado(nota):
    if nota >= 17.5:
        return "Destacado"
    elif nota >= 12.5:
        return "Promedio"
    else:
        return "Desaprobado"

In [0]:
estado_udf = F.udf(estado, StringType())

In [0]:
df_ciclo = spark.table(f"{catalogo}.{esquema_source}.ciclo")
df_programa = spark.table(f"{catalogo}.{esquema_source}.programa")
df_matricula = spark.table(f"{catalogo}.{esquema_source}.matricula")

In [0]:
df_matricula_pe_select = df_matricula.select("ACAD_CAREER",
                                             "STRM",
                                             "EMPLID",
                                             "EMPLID_NAME",
                                             "ACAD_PROG",
                                             "NBR_CLASS",
                                             "COURSE",
                                             "COURSE_DESC",
                                             "PROM"
                                             )

In [0]:
df_matricula_pe_distinct = df_matricula_pe_select.distinct()

In [0]:
df_matricula_pe_estado = df_matricula_pe_distinct.withColumn("ESTADO",estado_udf("PROM"))

In [0]:
df_matricula_pe_joined = df_matricula_pe_estado.alias("m").join(df_programa.alias("p"),col("m.ACAD_PROG") == col("p.ACAD_PROG"),"inner")\
                                                          .join(df_ciclo.alias("c"),col("c.STRM") == col("m.STRM"),"inner")

In [0]:
df_matricula_pe = df_matricula_pe_joined.select(col("m.ACAD_CAREER"),
                                                col("m.STRM"),
                                                col("c.DESCR"),
                                                col("m.EMPLID"),
                                                col("m.EMPLID_NAME"),
                                                col("m.ACAD_PROG"),
                                                col("p.DESCR"),
                                                col("m.NBR_CLASS"),
                                                col("m.COURSE"),
                                                col("m.COURSE_DESC"),
                                                col("m.PROM"),
                                                col("m.ESTADO"),
                                                ).orderBy("m.EMPLID","m.STRM").filter(col("m.ACAD_CAREER") == "PRGS")

In [0]:
df_matricula_pe.write.mode("overwrite").insertInto(f"{catalogo}.{esquema_sink}.programa_estatus_calif")